In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 03 · Auth Manager: brokering credentials per call

**Primer sections:** §4.3 scoped credential per tool call · §5 secrets and credential handling.

An agent should never hold one wide, long-lived token. Instead it authenticates to a **broker**
with its own identity and receives, per tool call, a credential scoped to that tool: right audience,
minimal scope, short TTL, and — for delegated calls — the user's identity with the agent as actor.
Google's broker is **Auth Manager** (part of Agent Identity): it vaults API keys, 2-legged OAuth
client credentials and end-user 3-legged OAuth tokens as *auth providers*, gated by IAM
(`roles/agentidentity.user` on the provider). `LocalAuthManager` reproduces its API
(`retrieveCredentials` → success | pending | uri_consent_required | consent_rejected, and
`credentials:finalize`) in memory so the very same ADK tool code runs offline.

In [ ]:
import logging

from agentsec.logging_utils import quiet_logs

quiet_logs(logging.ERROR)

from urllib.parse import parse_qs, urlsplit

import jwt  # display only

from agentsec.agents import LocalStack, Step, reset_demo_state
from agentsec.config import Settings
from agentsec.identity import (
    AgentIdentity,
    LocalAuthManager,
    PermissionDenied,
    ProviderKind,
    TokenIssuer,
)
from agentsec.runtime import resume_after_auth, run_turn, seed_session
from agentsec.secrets import SecretValue

reset_demo_state()

def peek(token: str) -> dict:
    return jwt.decode(token, options={"verify_signature": False})

def short(spiffe: str) -> str:
    return spiffe.rsplit("/", 1)[-1]

ORG, PROJECT = "123456789012", "987654321098"
agent = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="support-agent", org_id=ORG)
other_agent = AgentIdentity.for_agent_engine(project_number="111111111111", location="us-central1", engine_id="marketing-agent", org_id=ORG)
issuer = TokenIssuer()
am = LocalAuthManager(issuer, project="demo-project", location="global")

## 1. Three kinds of provider

* **3-legged OAuth** — *delegated*: the user consents once; the agent later retrieves a user-scoped
  token and never sees the refresh token.
* **2-legged OAuth** — the agent's *own* authority against a SaaS API (client credentials).
* **API key** — own authority; the key lives in the vault as a `SecretValue`.

In [ ]:
crm = am.create_provider(
    "crm-3lo", ProviderKind.THREE_LEGGED_OAUTH, audience="https://crm.acme.example",
    allowed_scopes=("crm.read", "crm.write"),
    authorization_url="https://idp.acme.example/o/oauth2/auth", token_url="https://idp.acme.example/o/oauth2/token",
    client_id="acme-support-agent", client_secret=SecretValue("s3cr3t", name="crm-client-secret"),
)
erp = am.create_provider(
    "erp-2lo", ProviderKind.TWO_LEGGED_OAUTH, audience="https://erp.acme.example",
    allowed_scopes=("erp.read",), client_id="erp-client", token_url="https://erp.acme.example/token",
)
weather = am.create_provider(
    "weather-key", ProviderKind.API_KEY, audience="https://weather.example",
    api_key=SecretValue("k-123-weather", name="weather-api-key"),
)
for name in am.list_providers():
    print(name)
print("\nclient secret in the vault prints as:", crm.client_secret)   # never the plaintext

## 2. IAM on the provider: who may retrieve credentials

The agent's principal (or a `principalSet` for a fleet) gets `roles/agentidentity.user` on the
provider. No binding → `PermissionDenied`, and the attempt is still logged. Scopes not allowed on
the provider are refused too — the provider is the outer envelope for everything minted through it.

In [ ]:
am.add_iam_policy_binding(crm.name, agent.iam_principal)                                  # this one agent
am.add_iam_policy_binding(weather.name, f"principalSet://agents.global.org-{ORG}.system.id.goog/attribute.platformContainer/aiplatform/projects/{PROJECT}")  # every agent in the project
# erp-2lo deliberately gets no binding

def attempt(label, **kw):
    try:
        r = am.retrieve_credentials(**kw)
        print(f"{label:<52} {r.kind}")
    except PermissionDenied as e:
        print(f"{label:<52} PermissionDenied: {e}")

attempt("marketing-agent → crm-3lo (no binding)", auth_provider=crm.name, user_id="u-ana", caller=other_agent, scopes=["crm.read"])
attempt("support-agent → crm-3lo, scope crm.admin (not allowed)", auth_provider=crm.name, user_id="u-ana", caller=agent, scopes=["crm.admin"])
attempt("support-agent → erp-2lo (no binding)", auth_provider=erp.name, user_id=None, caller=agent)
attempt("marketing-agent → weather-key (other project)", auth_provider=weather.name, user_id=None, caller=other_agent)
attempt("support-agent → weather-key (principalSet)", auth_provider=weather.name, user_id=None, caller=agent)

## 3. The 3-legged flow: `uri_consent_required` → consent → `finalize` → `success`

First retrieval for a user returns a consent URI (with PKCE, state = consent nonce, the broker's
callback as redirect URI). The front-end sends the user there; after the redirect it calls
`credentials:finalize` with the nonce and a validation state. From then on the agent gets a
delegated token per call.

In [ ]:
r1 = am.retrieve_credentials(auth_provider=crm.name, user_id="u-ana", caller=agent, scopes=["crm.read"],
                             continue_uri="https://app.acme.example/validateUserId")
print("outcome:", r1.kind)
q = parse_qs(urlsplit(r1.authorization_uri).query)
print("consent URI params:", {k: v[0] for k, v in q.items()})
assert r1.kind == "uri_consent_required" and q["state"][0] == r1.consent_nonce

# ... the user consents at the IdP; the front-end finalises with the broker ...
am.finalize(auth_provider=crm.name, user_id="u-ana", consent_nonce=r1.consent_nonce, user_id_validation_state="VALIDATED")

r2 = am.retrieve_credentials(auth_provider=crm.name, user_id="u-ana", caller=agent, scopes=["crm.read"])
print("\noutcome:", r2.kind, "| header:", r2.header)
claims = issuer.verify(r2.token, audience="https://crm.acme.example")
print(f"token: sub={claims.subject} act={short(claims.actor)} scope={claims.scopes} authority={claims.raw['authority']}")
assert claims.subject == "u-ana" and claims.actor == agent.spiffe_id and claims.scopes == {"crm.read"}

Consent is per (provider, user) and can be **rejected**; a rejected consent yields
`consent_rejected` instead of a token, and ADK surfaces that as a tool error.

In [ ]:
r_ben = am.retrieve_credentials(auth_provider=crm.name, user_id="u-ben", caller=agent, scopes=["crm.read"])
am.finalize(auth_provider=crm.name, user_id="u-ben", consent_nonce=r_ben.consent_nonce, user_id_validation_state="REJECTED")
r_ben2 = am.retrieve_credentials(auth_provider=crm.name, user_id="u-ben", caller=agent, scopes=["crm.read"])
print("u-ben after rejecting consent:", r_ben2.kind)
assert r_ben2.kind == "consent_rejected" and r_ben2.token is None

## 4. 2-legged and API-key providers are the agent's **own** authority

No user is involved; the token/key is attributable to the agent only. Note the API key is returned
for the request header at the single point of use, and its plaintext never appears in the access log.

In [ ]:
am.add_iam_policy_binding(erp.name, agent.iam_principal)
r_erp = am.retrieve_credentials(auth_provider=erp.name, user_id=None, caller=agent)
erp_claims = peek(r_erp.token)
print(f"2LO token: sub={short(erp_claims['sub'])} aud={erp_claims['aud']} scope={erp_claims['scope']} authority={erp_claims['authority']}")

r_key = am.retrieve_credentials(auth_provider=weather.name, user_id=None, caller=agent)
print(f"API key  : header={r_key.header} token={r_key.token}")
assert r_key.header == "X-API-Key" and r_key.token == "k-123-weather"
assert "k-123-weather" not in repr(am.access_log)

## 5. The access log: attributable to the agent **and** the user

Every retrieval is recorded with the caller's SPIFFE ID, the user (for 3LO), the provider, the
scopes, and the outcome — including the denials. This is the property Cloud Audit Logs give you on
GCP: dual identity for delegated access, single identity for own-authority access.

In [ ]:
print(f"{'agent':<16} {'user':<8} {'provider':<14} {'scopes':<22} outcome")
for e in am.access_log:
    print(f"{short(e.agent):<16} {e.user or '-':<8} {e.provider.rsplit('/', 1)[-1]:<14} {','.join(e.scopes) or '-':<22} {e.outcome}")

## 6. The ADK path: `crm_lookup` → `adk_request_credential` → finalize → resume

`LocalStack` wires an Auth-Manager-backed `AuthenticatedFunctionTool` (`crm_lookup`) into the
reference agent through ADK's `GcpAuthProviderScheme`. When the model calls the tool and no consent
exists, ADK emits an `adk_request_credential` function call instead of running the tool. The front-end
sends the user to the consent URI, finalises with the broker, and resumes the run; ADK re-invokes the
tool with the credential injected. The tool code never touches a refresh token or client secret.

In [ ]:
stack = LocalStack.create(Settings())
await seed_session(stack.runner, user_id="u-ana", session_id="s1",
                   user={"subject": "u-ana", "email": "ana@customer.example", "tenant": "acme"},
                   scopes=["customers:read", "orders:read"])

stack.script(Step.call("crm_lookup", email="ana@customer.example"), Step.say("Here is the CRM record."))
r = await run_turn(stack.runner, user_id="u-ana", session_id="s1", message="check ana in the CRM")
print(r.summary())
assert len(r.pending_auth) == 1 and not any(t["name"] == "crm_lookup" and "content" in t["response"] for t in r.tool_responses)

pending = r.pending_auth[0]
print("\nconsent URI:", pending.auth_uri[:60] + "…")
print("nonce      :", pending.consent_nonce)

In [ ]:
# The front-end redirects the user, the IdP calls back into the broker, the front-end finalises:
stack.auth_manager.finalize(auth_provider=stack.crm_provider, user_id="u-ana", consent_nonce=pending.consent_nonce)

stack.script(Step.say("Here is the CRM record."))
r2 = await resume_after_auth(stack.runner, user_id="u-ana", session_id="s1", pending=pending)
crm = next(t["response"] for t in r2.tool_responses if t["name"] == "crm_lookup")
print(crm["content"])
assert "user-delegated token" in crm["content"]

print("\nbroker log:", [(e.outcome, e.user, short(e.agent)) for e in stack.auth_manager.access_log])

## On Google Cloud

* Providers: `gcloud agent-identity auth-providers create …` or Terraform
  `google_agent_identity_auth_provider` (`infra/terraform/auth_provider.tf`); bind
  `roles/agentidentity.user` to the agent principal on the provider.
* ADK: register `GcpAuthProvider()` once, attach `GcpAuthProviderScheme(name=…, scopes=…)` to an
  `McpToolset` or `AuthenticatedFunctionTool`; ADK fetches per call and injects.
* Two secret-handling paths: **direct** (ADK attaches the credential in the agent process — simple,
  but a hijacked runtime can read it for its lifetime) vs **gateway** (Auth Manager encrypts, Agent
  Gateway decrypts and injects at egress — the agent code never sees the raw credential).

**In one sentence:** "I design for no secrets first and vault what's left in Auth Manager
with per-provider IAM. The agent authenticates to the broker with its own identity and gets a
credential per call — audience-scoped, short-lived, and for user data a 3LO token that names both
the user and the agent, so the access log is attributable to both. Then I decide explicitly whether
the agent process may ever hold a raw user credential; if not, egress goes through Agent Gateway."